# 2.2 eGRID Region EDA

Exploratory data analysis for eGRID subregion and balancing authority outage statistics.

Includes:
- Frequency statistics for yearly outages, customer_hours, customer_hours_per_capita
- Seasonality analysis (by month)
- Time of day analysis (by hour)
- Balancing authority breakdown

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load county-level data (which includes eGRID enrichments)
data_dir = Path("../../data/processed/introduction")

with open(data_dir / "county_year_summary.json") as f:
    county_summary = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_month.json") as f:
    county_month = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_hour.json") as f:
    county_hour = pd.DataFrame(json.load(f))

print(f"County summary records: {len(county_summary):,}")

In [ ]:
# View eGRID-related columns
print("eGRID-related columns:")
print(county_summary[['egrid_subregion', 'balancing_authority']].head(10))

In [ ]:
# Unique eGRID subregions and balancing authorities
print(f"Unique eGRID subregions: {county_summary['egrid_subregion'].nunique()}")
print(f"Unique balancing authorities: {county_summary['balancing_authority'].nunique()}")
print(f"\neGRID subregions: {sorted(county_summary['egrid_subregion'].dropna().unique())}")

In [ ]:
print(f"\nBalancing authorities: {sorted(county_summary['balancing_authority'].dropna().unique())}")

## eGRID Subregion Statistics

Aggregate outage data by eGRID subregion.

In [ ]:
# Aggregate by eGRID subregion
egrid_totals = county_summary.groupby('egrid_subregion').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum',
    'fips': 'nunique',
    'state': 'nunique'
}).rename(columns={'fips': 'counties_count', 'state': 'states_count'}).reset_index()

# Calculate customer hours per capita
egrid_totals['customer_hours_per_capita'] = egrid_totals['customer_hours'] / egrid_totals['population']

egrid_totals = egrid_totals.sort_values('customer_hours', ascending=False)

print(f"Total eGRID subregions: {len(egrid_totals)}")
egrid_totals

In [ ]:
# Descriptive statistics for eGRID-level data
egrid_totals[['outage_count', 'customer_hours', 'customer_hours_per_capita', 'counties_count']].describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Outage count by eGRID subregion
axes[0, 0].barh(egrid_totals['egrid_subregion'], egrid_totals['outage_count'], color='steelblue')
axes[0, 0].set_title('Total Outages by eGRID Subregion')
axes[0, 0].set_xlabel('Outage Count')
axes[0, 0].set_ylabel('eGRID Subregion')

# Customer hours by eGRID subregion
axes[0, 1].barh(egrid_totals['egrid_subregion'], egrid_totals['customer_hours'] / 1e6, color='coral')
axes[0, 1].set_title('Total Customer Hours by eGRID Subregion')
axes[0, 1].set_xlabel('Customer Hours (millions)')
axes[0, 1].set_ylabel('eGRID Subregion')

# Customer hours per capita by eGRID subregion
egrid_sorted_capita = egrid_totals.sort_values('customer_hours_per_capita', ascending=True)
axes[1, 0].barh(egrid_sorted_capita['egrid_subregion'], egrid_sorted_capita['customer_hours_per_capita'], color='teal')
axes[1, 0].set_title('Customer Hours per Capita by eGRID Subregion')
axes[1, 0].set_xlabel('Customer Hours per Capita')
axes[1, 0].set_ylabel('eGRID Subregion')

# Counties count by eGRID subregion
axes[1, 1].barh(egrid_totals['egrid_subregion'], egrid_totals['counties_count'], color='purple')
axes[1, 1].set_title('Number of Counties by eGRID Subregion')
axes[1, 1].set_xlabel('Number of Counties')
axes[1, 1].set_ylabel('eGRID Subregion')

plt.tight_layout()
plt.show()

## Balancing Authority Statistics

In [ ]:
# Aggregate by balancing authority
ba_totals = county_summary.groupby('balancing_authority').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum',
    'fips': 'nunique',
    'egrid_subregion': 'nunique'
}).rename(columns={'fips': 'counties_count', 'egrid_subregion': 'subregions_count'}).reset_index()

# Calculate customer hours per capita
ba_totals['customer_hours_per_capita'] = ba_totals['customer_hours'] / ba_totals['population']

print(f"Total balancing authorities: {len(ba_totals)}")
ba_totals.sort_values('customer_hours', ascending=False).head(20)

In [ ]:
# Top 20 balancing authorities by outage count
top_ba_outages = ba_totals.nlargest(20, 'outage_count')

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top_ba_outages['balancing_authority'][::-1], 
        top_ba_outages['outage_count'][::-1], 
        color='steelblue')
ax.set_title('Top 20 Balancing Authorities by Outage Count')
ax.set_xlabel('Outage Count')
ax.set_ylabel('Balancing Authority')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 balancing authorities by customer hours
top_ba_hours = ba_totals.nlargest(20, 'customer_hours')

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top_ba_hours['balancing_authority'][::-1], 
        top_ba_hours['customer_hours'][::-1] / 1e6, 
        color='coral')
ax.set_title('Top 20 Balancing Authorities by Customer Hours')
ax.set_xlabel('Customer Hours (millions)')
ax.set_ylabel('Balancing Authority')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 balancing authorities by customer hours per capita (min population threshold)
min_pop = 500000  # At least 500k population
ba_filtered = ba_totals[ba_totals['population'] >= min_pop]
top_ba_per_capita = ba_filtered.nlargest(20, 'customer_hours_per_capita')

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top_ba_per_capita['balancing_authority'][::-1], 
        top_ba_per_capita['customer_hours_per_capita'][::-1], 
        color='teal')
ax.set_title(f'Top 20 Balancing Authorities by Customer Hours per Capita\n(min population: {min_pop:,})')
ax.set_xlabel('Customer Hours per Capita')
ax.set_ylabel('Balancing Authority')
plt.tight_layout()
plt.show()

### Distribution of eGRID/BA Level Metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# BA outage count distribution
axes[0, 0].hist(ba_totals['outage_count'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of BA Outage Count')
axes[0, 0].set_xlabel('Outage Count')
axes[0, 0].set_ylabel('Frequency')

axes[1, 0].hist(np.log10(ba_totals['outage_count'] + 1), bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of BA Outage Count (log10)')
axes[1, 0].set_xlabel('log10(Outage Count)')
axes[1, 0].set_ylabel('Frequency')

# BA customer hours distribution
axes[0, 1].hist(ba_totals['customer_hours'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of BA Customer Hours')
axes[0, 1].set_xlabel('Customer Hours')
axes[0, 1].set_ylabel('Frequency')

axes[1, 1].hist(np.log10(ba_totals['customer_hours'] + 1), bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_title('Distribution of BA Customer Hours (log10)')
axes[1, 1].set_xlabel('log10(Customer Hours)')
axes[1, 1].set_ylabel('Frequency')

# BA customer hours per capita distribution
axes[0, 2].hist(ba_totals['customer_hours_per_capita'], bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[0, 2].set_title('Distribution of BA Customer Hours per Capita')
axes[0, 2].set_xlabel('Customer Hours per Capita')
axes[0, 2].set_ylabel('Frequency')

axes[1, 2].hist(np.log10(ba_totals['customer_hours_per_capita'] + 0.001), bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[1, 2].set_title('Distribution of BA Customer Hours per Capita (log10)')
axes[1, 2].set_xlabel('log10(Customer Hours per Capita)')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Percentile analysis for BA metrics
percentiles = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
metrics = ['outage_count', 'customer_hours', 'customer_hours_per_capita', 'counties_count']

percentile_df = ba_totals[metrics].quantile(percentiles)
percentile_df.index = [f"{int(p*100)}%" for p in percentiles]
print("Percentile distribution for balancing authority metrics:")
percentile_df

## eGRID Subregion Yearly Trends

In [ ]:
# Aggregate by eGRID subregion and year
egrid_yearly = county_summary.groupby(['egrid_subregion', 'year']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum'
}).reset_index()

egrid_yearly['customer_hours_per_capita'] = egrid_yearly['customer_hours'] / egrid_yearly['population']

print(f"eGRID-year combinations: {len(egrid_yearly)}")
egrid_yearly.head(10)

In [ ]:
# Heatmap of outages by eGRID subregion and year
egrid_yearly_pivot = egrid_yearly.pivot(index='egrid_subregion', columns='year', values='outage_count')

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(egrid_yearly_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Outage Count by eGRID Subregion and Year')
ax.set_xlabel('Year')
ax.set_ylabel('eGRID Subregion')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of customer hours by eGRID subregion and year
egrid_hours_pivot = egrid_yearly.pivot(index='egrid_subregion', columns='year', values='customer_hours') / 1e6

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(egrid_hours_pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Customer Hours (millions) by eGRID Subregion and Year')
ax.set_xlabel('Year')
ax.set_ylabel('eGRID Subregion')
plt.tight_layout()
plt.show()

In [ ]:
# Get top 5 eGRID subregions by total outages and show yearly trends
top_5_egrid = egrid_totals.nlargest(5, 'outage_count')['egrid_subregion'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for egrid in top_5_egrid:
    egrid_data = egrid_yearly[egrid_yearly['egrid_subregion'] == egrid]
    axes[0].plot(egrid_data['year'], egrid_data['outage_count'], marker='o', label=egrid)
    axes[1].plot(egrid_data['year'], egrid_data['customer_hours'] / 1e6, marker='o', label=egrid)

axes[0].set_title('Yearly Outage Trend - Top 5 eGRID Subregions')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Outage Count')
axes[0].legend()

axes[1].set_title('Yearly Customer Hours Trend - Top 5 eGRID Subregions')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Customer Hours (millions)')
axes[1].legend()

plt.tight_layout()
plt.show()

## Seasonality by eGRID Subregion

In [ ]:
# Merge county_month with county_summary to get eGRID subregion
county_month_enriched = county_month.merge(
    county_summary[['fips', 'year', 'egrid_subregion']].drop_duplicates(),
    on=['fips', 'year'],
    how='left'
)

# Aggregate by eGRID subregion and month
egrid_monthly = county_month_enriched.groupby(['egrid_subregion', 'month']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
egrid_monthly['month_name'] = egrid_monthly['month'].apply(lambda x: month_names[x-1])

egrid_monthly.head()

In [ ]:
# Heatmap of outages by eGRID subregion and month
egrid_monthly_pivot = egrid_monthly.pivot(index='egrid_subregion', columns='month', values='outage_count')

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(egrid_monthly_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Outage Count by eGRID Subregion and Month')
ax.set_xlabel('Month')
ax.set_ylabel('eGRID Subregion')
ax.set_xticklabels(month_names)
plt.tight_layout()
plt.show()

In [ ]:
# Normalize by eGRID subregion total to see seasonal patterns
egrid_totals_monthly = egrid_monthly.groupby('egrid_subregion')['outage_count'].transform('sum')
egrid_monthly['outage_pct'] = egrid_monthly['outage_count'] / egrid_totals_monthly * 100

# Top 6 eGRID subregions for line chart
top_6_egrid = egrid_totals.nlargest(6, 'outage_count')['egrid_subregion'].tolist()

fig, ax = plt.subplots(figsize=(14, 6))
for egrid in top_6_egrid:
    data = egrid_monthly[egrid_monthly['egrid_subregion'] == egrid]
    ax.plot(data['month'], data['outage_pct'], marker='o', label=egrid)

ax.set_title('Seasonal Pattern by eGRID Subregion (% of Annual Total)')
ax.set_xlabel('Month')
ax.set_ylabel('Percentage of Annual Outages')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.legend(title='eGRID Subregion')
plt.tight_layout()
plt.show()

## Time of Day by eGRID Subregion

In [ ]:
# Merge county_hour with county_summary to get eGRID subregion
county_hour_enriched = county_hour.merge(
    county_summary[['fips', 'year', 'egrid_subregion']].drop_duplicates(),
    on=['fips', 'year'],
    how='left'
)

# Aggregate by eGRID subregion and hour
egrid_hourly = county_hour_enriched.groupby(['egrid_subregion', 'hour']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

egrid_hourly.head()

In [ ]:
# Heatmap of outages by eGRID subregion and hour
egrid_hourly_pivot = egrid_hourly.pivot(index='egrid_subregion', columns='hour', values='outage_count')

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(egrid_hourly_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, annot_kws={'size': 7})
ax.set_title('Outage Count by eGRID Subregion and Hour')
ax.set_xlabel('Hour')
ax.set_ylabel('eGRID Subregion')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize by eGRID subregion total
egrid_totals_hourly = egrid_hourly.groupby('egrid_subregion')['outage_count'].transform('sum')
egrid_hourly['outage_pct'] = egrid_hourly['outage_count'] / egrid_totals_hourly * 100

fig, ax = plt.subplots(figsize=(14, 6))
for egrid in top_6_egrid:
    data = egrid_hourly[egrid_hourly['egrid_subregion'] == egrid]
    ax.plot(data['hour'], data['outage_pct'], marker='o', markersize=4, label=egrid)

ax.set_title('Hourly Pattern by eGRID Subregion (% of Total)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Percentage of Outages')
ax.set_xticks(range(0, 24))
ax.legend(title='eGRID Subregion')
plt.tight_layout()
plt.show()

In [ ]:
# Define time of day periods
def categorize_hour(hour):
    if 6 <= hour < 12:
        return 'Morning (6-11)'
    elif 12 <= hour < 18:
        return 'Afternoon (12-17)'
    elif 18 <= hour < 22:
        return 'Evening (18-21)'
    else:
        return 'Night (22-5)'

egrid_hourly['time_period'] = egrid_hourly['hour'].apply(categorize_hour)

# Aggregate by eGRID subregion and time period
egrid_period = egrid_hourly.groupby(['egrid_subregion', 'time_period']).agg({
    'outage_count': 'sum'
}).reset_index()

# Pivot for stacked bar chart
time_period_order = ['Morning (6-11)', 'Afternoon (12-17)', 'Evening (18-21)', 'Night (22-5)']
egrid_period_pivot = egrid_period.pivot(index='egrid_subregion', columns='time_period', values='outage_count')
egrid_period_pivot = egrid_period_pivot[time_period_order]

fig, ax = plt.subplots(figsize=(14, 8))
egrid_period_pivot.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Outages by Time of Day - eGRID Subregions')
ax.set_xlabel('Outage Count')
ax.set_ylabel('eGRID Subregion')
ax.legend(title='Time Period', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Balancing Authority Yearly Trends

In [ ]:
# Aggregate by balancing authority and year
ba_yearly = county_summary.groupby(['balancing_authority', 'year']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum'
}).reset_index()

ba_yearly['customer_hours_per_capita'] = ba_yearly['customer_hours'] / ba_yearly['population']

# Get top 5 balancing authorities by total outages
top_5_ba = ba_totals.nlargest(5, 'outage_count')['balancing_authority'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ba in top_5_ba:
    ba_data = ba_yearly[ba_yearly['balancing_authority'] == ba]
    axes[0].plot(ba_data['year'], ba_data['outage_count'], marker='o', label=ba)
    axes[1].plot(ba_data['year'], ba_data['customer_hours'] / 1e6, marker='o', label=ba)

axes[0].set_title('Yearly Outage Trend - Top 5 Balancing Authorities')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Outage Count')
axes[0].legend(fontsize=8)

axes[1].set_title('Yearly Customer Hours Trend - Top 5 Balancing Authorities')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Customer Hours (millions)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()